In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/srujanakedigehalli/transcripts/transcripts.csv


In [2]:
# ── Cell 1: Install Dependencies ───────────────────────────────────────────
!pip install sentence-transformers faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.0 MB/s eta 0:00:00:00:0100:01


In [3]:
# ── Cell 2: Load Transcripts ───────────────────────────────────────────────
import pandas as pd

# Update this path to wherever your CSV is in Kaggle input
df = pd.read_csv('/kaggle/input/datasets/srujanakedigehalli/transcripts/transcripts.csv')

print(f"Transcripts loaded : {len(df)}")
print(f"Columns            : {df.columns.tolist()}")
print(f"Categories covered : {df['category'].value_counts().to_dict()}")

Transcripts loaded : 55
Columns            : ['video_id', 'title', 'publishedAt', 'channelId', 'channelTitle', 'categoryId', 'trending_date', 'tags', 'view_count', 'likes', 'dislikes', 'comment_count', 'thumbnail_link', 'comments_disabled', 'ratings_disabled', 'description', 'category', 'country', 'engagement_ratio', 'hook', 'body', 'full_transcript', 'word_count']
Categories covered : {'Autos & Vehicles': 39, 'Comedy': 16}


In [4]:
# ── Cell 3: Create Chunks ──────────────────────────────────────────────────
# Each video produces 2 chunks — hook (0-10s) and body (10-60s)
# We keep them separate because hook and body serve different purposes

chunks = []

for _, row in df.iterrows():
    for seg_type in ['hook', 'body']:
        text = str(row.get(seg_type, '')).strip()

        # Skip if empty or too short to be meaningful
        if len(text) < 20:
            continue

        chunks.append({
            "chunk_id"        : f"{row['video_id']}_{seg_type}",
            "video_id"        : row['video_id'],
            "text"            : text,
            "segment_type"    : seg_type,
            "title"           : row['title'],
            "category"        : row['category'],
            "view_count"      : row['view_count'],
            "engagement_ratio": row['engagement_ratio'],
            "country"         : row['country']
        })

print(f"Total chunks created : {len(chunks)}")
print(f"  Hook chunks        : {sum(1 for c in chunks if c['segment_type'] == 'hook')}")
print(f"  Body chunks        : {sum(1 for c in chunks if c['segment_type'] == 'body')}")

# Preview a hook chunk
hook_sample = next(c for c in chunks if c['segment_type'] == 'hook')
print(f"\nSample hook chunk:")
print(f"  Title   : {hook_sample['title']}")
print(f"  Text    : {hook_sample['text'][:150]}")
print(f"  Category: {hook_sample['category']}")
print(f"  Engagement: {hook_sample['engagement_ratio']:.4f}")

Total chunks created : 103
  Hook chunks        : 48
  Body chunks        : 55

Sample hook chunk:
  Title   : In Loving Memory of my Dad.
  Text    : we're headed to john wayne airport to pick up one of the most important people in my life my dad it's going to be violent dad here we go
  Category: Autos & Vehicles
  Engagement: 0.2423


In [5]:
# ── Cell 4: Generate Embeddings ────────────────────────────────────────────
import numpy as np
from sentence_transformers import SentenceTransformer

# all-MiniLM-L6-v2 is fast, lightweight, and works well for semantic search
model = SentenceTransformer('all-MiniLM-L6-v2', device = 'cpu')

texts      = [c['text'] for c in chunks]
embeddings = model.encode(texts, batch_size=32, show_progress_bar=True)
embeddings = np.array(embeddings).astype('float32')

print(f"\nEmbedding shape: {embeddings.shape}")
# Should be (num_chunks, 384) — 384 is the MiniLM embedding dimension

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]


Embedding shape: (103, 384)


In [6]:
pip install faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [7]:
# ── Cell 5: Build FAISS Index ──────────────────────────────────────────────
import faiss

index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

print(f"FAISS index built")
print(f"Total vectors stored: {index.ntotal}")

FAISS index built
Total vectors stored: 103


In [8]:
# ── Cell 6: Test Retrieval ─────────────────────────────────────────────────
def retrieve(query, category=None, segment_type='hook', top_k=5):
    """
    Given a query string, retrieve the most semantically similar
    chunks from the FAISS index.
    
    Args:
        query        : what you're searching for e.g. "how to study"
        category     : filter by category e.g. "Education" (optional)
        segment_type : 'hook' or 'body'
        top_k        : how many results to return
    """
    query_vec          = model.encode([query]).astype('float32')
    distances, indices = index.search(query_vec, top_k * 4)  # over-fetch then filter

    results = [chunks[i] for i in indices[0]]

    # Apply filters
    if segment_type:
        results = [r for r in results if r['segment_type'] == segment_type]
    if category:
        results = [r for r in results if r['category'] == category]

    return results[:top_k]


# ── Test it ────────────────────────────────────────────────────────────────
print("=" * 60)
print("TEST 1: General hook search")
print("=" * 60)
results = retrieve("how to make money online", segment_type='hook', top_k=3)
for r in results:
    print(f"\nTitle      : {r['title']}")
    print(f"Category   : {r['category']}")
    print(f"Engagement : {r['engagement_ratio']:.4f}")
    print(f"Hook text  : {r['text'][:200]}")

print("\n" + "=" * 60)
print("TEST 2: Category filtered search")
print("=" * 60)

# Use a category that exists in your 55 transcripts
available_categories = list(set(c['category'] for c in chunks))
print(f"Available categories: {available_categories}")

test_category = available_categories[0]
results = retrieve("top tips", category=test_category,
                   segment_type='hook', top_k=3)
for r in results:
    print(f"\nTitle      : {r['title']}")
    print(f"Hook text  : {r['text'][:200]}")

TEST 1: General hook search

Title      : Building A Laser Baby
Category   : Comedy
Engagement : 0.1752
Hook text  : this is going to be a weird video let me give you some context this video is sponsored by amazon yeah i don't know how either some poor guy at the marketing team was like i know the perfect person for

Title      : If the 'Forgot your password' thing was a person
Category   : Comedy
Engagement : 0.1914
Hook text  : how you doing hey welcome here uh so i wanted to take a look at all the reward points that i had on my account oh okay you plan on using them soon oh yeah i want to get that new tablet that came

Title      : I Built A Boat, Then Sailed The Ocean...
Category   : Comedy
Engagement : 0.1944
Hook text  : jack do you know where we're going today jack no you just haven't told me you did not tell me what we're doing what a boat jack what do you mean you bought a boat i brought a boat look at how sunny it

TEST 2: Category filtered search
Available categories: ['Auto

In [9]:
# ── Cell 7: Save Everything ────────────────────────────────────────────────
import pickle, faiss

# Save FAISS index
faiss.write_index(index, '/kaggle/working/hooks.index')

# Save chunks metadata (needed to map index positions back to video info)
with open('/kaggle/working/chunks_metadata.pkl', 'wb') as f:
    pickle.dump(chunks, f)

# Save chunks as CSV too (easier to inspect)
chunks_df = pd.DataFrame(chunks)
chunks_df.to_csv('/kaggle/working/chunks.csv', index=False)

print("Saved:")
print("  hooks.index          — FAISS vector index")
print("  chunks_metadata.pkl  — chunk metadata (title, category, engagement etc.)")
print("  chunks.csv           — human-readable version of chunks")
print(f"\nDownload all 3 files from the Output tab and upload to the next notebook")

Saved:
  hooks.index          — FAISS vector index
  chunks_metadata.pkl  — chunk metadata (title, category, engagement etc.)
  chunks.csv           — human-readable version of chunks

Download all 3 files from the Output tab and upload to the next notebook
